In [1]:
import pandas as pd
from google.cloud import bigquery

In [6]:
from google.oauth2 import service_account
from google.cloud import bigquery

credentials = service_account.Credentials.from_service_account_file(
    r"E:\retail-pulse-project\Credential\retail-pulse-496303-ca384c444638.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="retail-pulse-496303"
)

print("Connected to BigQuery!")

Connected to BigQuery!


In [14]:
import pandas as pd

# -----------------------------
# RFM QUERY
# -----------------------------

rfm_query = """
SELECT
    customer_name,
    DATE_DIFF(
        CURRENT_DATE(),
        DATE(MAX(order_date)),
        DAY
    ) AS recency_days,

    COUNT(DISTINCT order_id) AS frequency,

    ROUND(SUM(sales), 2) AS monetary

FROM raw_data.fact_sales

GROUP BY customer_name
"""

# -----------------------------
# RUN QUERY
# -----------------------------

rfm = client.query(
    rfm_query,
    location="US"
).to_dataframe()

# -----------------------------
# CREATE RFM SCORES
# -----------------------------

rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    5,
    labels=[5,4,3,2,1]
).astype(int)

rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
).astype(int)

rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    5,
    labels=[1,2,3,4,5]
).astype(int)

# -----------------------------
# TOTAL SCORE
# -----------------------------

rfm['RFM_total'] = (
    rfm['R_score'] +
    rfm['F_score'] +
    rfm['M_score']
)

# -----------------------------
# SEGMENT FUNCTION
# -----------------------------

def segment(score):

    if score >= 13:
        return "Champions"

    elif score >= 10:
        return "Loyal Customers"

    elif score >= 7:
        return "At Risk"

    else:
        return "Lost"

# Apply segmentation
rfm['segment'] = rfm['RFM_total'].apply(segment)

# -----------------------------
# SHOW RESULT
# -----------------------------

print(rfm.head())

# -----------------------------
# UPLOAD TO BIGQUERY
# -----------------------------

table_id = "raw_data.customer_rfm"

job = client.load_table_from_dataframe(
    rfm,
    table_id,
    location="US"
)

job.result()

print("RFM table uploaded successfully!")

     customer_name  recency_days  frequency  monetary  R_score  F_score  \
0    Aaron Bergman          4170         37   24646.0        3        4   
1    Aaron Hawkins          4166         34   20759.0        4        3   
2   Aaron Smayling          4177         31   14207.0        2        2   
3  Adam Bellavance          4189         41   20189.0        2        5   
4        Adam Hart          4156         42   21720.0        5        5   

   M_score  RFM_total          segment  
0        5         12  Loyal Customers  
1        5         12  Loyal Customers  
2        3          7          At Risk  
3        5         12  Loyal Customers  
4        5         15        Champions  


C:\Users\ashwi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


RFM table uploaded successfully!
